# Bob — Polaris STS AssumeRole demo

Bob's catalog role has table-scoped grants on `demo.order_items` (shared with
Alice), `demo.products`, and `demo.events`. He has **no** grant on
`demo.customers` or `demo.orders` — the last cell proves that matters.

In [ ]:
import requests
from dotenv import dotenv_values

creds = dotenv_values('/home/jovyan/creds/bob.env')
CLIENT_ID = creds['POLARIS_CLIENT_ID']
CLIENT_SECRET = creds['POLARIS_CLIENT_SECRET']
CATALOG_NAME = creds['POLARIS_CATALOG_NAME']
REALM = creds['POLARIS_REALM']
assert CLIENT_ID != 'CHANGE_ME', 'polaris-setup has not run yet — see ~/bw/sts/runbook.md step 6/7'

# Separate, narrowly-scoped VASTDB key — NOT vended by Polaris (Generic
# Tables are metadata-only in Polaris 1.7.0). See runbook.md's VASTDB
# extension section for why these are two unrelated credential paths.
db_creds = dotenv_values('/home/jovyan/creds/bob_db.env')
DB_ACCESS_KEY = db_creds['VASTDB_ACCESS_KEY_ID']
DB_SECRET_KEY = db_creds['VASTDB_SECRET_ACCESS_KEY']
assert DB_ACCESS_KEY != 'CHANGE_ME', 'bob-db VAST key not filled in yet — see runbook.md VASTDB extension, VMS step'

# Polaris token for the REST calls in the next cell. Spark gets the same
# credential separately via spark.sql.catalog.polaris.credential.
POLARIS = 'http://polaris:8181'
TOKEN = requests.post(f'{POLARIS}/api/catalog/v1/oauth/tokens', data={
    'grant_type': 'client_credentials',
    'client_id': CLIENT_ID, 'client_secret': CLIENT_SECRET,
    'scope': 'PRINCIPAL_ROLE:ALL'}).json()['access_token']
HEADERS = {'Authorization': f'Bearer {TOKEN}', 'Polaris-Realm': REALM}

print(f'Catalog: {CATALOG_NAME}, principal: bob ({CLIENT_ID})')

In [ ]:
# Ask Polaris which VASTDB tables exist and where they live, instead of
# hardcoding the bucket and schema here.
#
# This is an HTTP call, not a Spark catalog operation, and it has to be:
# Generic Tables are a different entity subtype from Iceberg tables, so
# the Iceberg REST listTables endpoint does not return them and
# `SHOW TABLES IN polaris.vastdb_demo` finds nothing. They have their own
# list/load endpoints — confirmed by reading the `endpoints` list that
# /api/catalog/v1/config returns, rather than trusting the OpenAPI spec.
GENERIC_TABLES = f'{POLARIS}/api/catalog/polaris/v1/{CATALOG_NAME}/namespaces/vastdb_demo/generic-tables'

listed = [i['name'] for i in requests.get(GENERIC_TABLES, headers=HEADERS).json()['identifiers']]

# Listing is gated by catalog-wide TABLE_LIST, so all three names show up
# here including the other principal's — names leak, same tradeoff as the
# Iceberg side. Loading each one is gated per table by TABLE_FULL_METADATA,
# which is where the grants actually bite: the 403s below name op
# LOAD_TABLE.
registered, denied = {}, []
for name in listed:
    r = requests.get(f'{GENERIC_TABLES}/{name}', headers=HEADERS)
    if r.status_code == 403:
        denied.append(name)
        continue
    r.raise_for_status()
    registered[name] = r.json()['table']['properties']

print('listed by Polaris    :', listed)
print('metadata granted     :', sorted(registered))
print('metadata denied (403):', denied)
assert registered, 'No VASTDB generic tables loadable — has polaris-setup run?'

# Bucket, schema and endpoints all come from the registration. Every
# loadable table reports the same cluster, so read it off the first.
props = next(iter(registered.values()))
VASTDB_ENDPOINT = props['vastdb.endpoint']
# Registrations made before setup.sh added this property fall back to the
# single control endpoint, which works but hits one CNode.
VASTDB_DATA_ENDPOINTS = props.get('vastdb.data_endpoints', VASTDB_ENDPOINT)
DB = f"ndb.`{props['vastdb.bucket']}`.`{props['vastdb.schema']}`"


def ndb(name):
    """Fully-qualified ndb identifier, built from Polaris's registration."""
    p = registered[name]
    return f"ndb.`{p['vastdb.bucket']}`.`{p['vastdb.schema']}`.{p['vastdb.table']}"


print(f'ndb schema {DB}, endpoint {VASTDB_ENDPOINT}, '
      f'{len(VASTDB_DATA_ENDPOINTS.split(","))} data endpoint(s)')

In [ ]:
from pyspark.sql import SparkSession

# NOTE: spark.driver.extraClassPath / spark.executor.extraClassPath for the
# VAST connector jars are set in the image's static spark-defaults.conf,
# NOT here — that config cannot take effect from SparkSession.builder in
# client/local mode (the driver JVM has already started by this point).

spark = (
    SparkSession.builder.appName('bob-sts-demo')
    .config('spark.sql.catalog.polaris', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.polaris.type', 'rest')
    .config('spark.sql.catalog.polaris.uri', 'http://polaris:8181/api/catalog')
    .config('spark.sql.catalog.polaris.warehouse', CATALOG_NAME)
    .config('spark.sql.catalog.polaris.credential', f'{CLIENT_ID}:{CLIENT_SECRET}')
    .config('spark.sql.catalog.polaris.scope', 'PRINCIPAL_ROLE:ALL')
    .config('spark.sql.catalog.polaris.header.X-Iceberg-Access-Delegation', 'vended-credentials')
    .config('spark.sql.defaultCatalog', 'polaris')
    # VAST DB ("ndb") catalog — a second, unrelated catalog in the same
    # session. Auth here is bob-db's own static key, not vended; the
    # endpoints come from the Polaris registration read in the cell above.
    .config('spark.sql.catalog.ndb', 'spark.sql.catalog.ndb.VastCatalog')
    .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,ndb.NDBSparkSessionExtension')
    .config('spark.ndb.endpoint', VASTDB_ENDPOINT)
    .config('spark.ndb.data_endpoints', VASTDB_DATA_ENDPOINTS)
    .config('spark.ndb.access_key_id', DB_ACCESS_KEY)
    .config('spark.ndb.secret_access_key', DB_SECRET_KEY)
    # Spark's local-mode default (4) trips the VAST connector's write-safety
    # guard ("Unsafe Spark configuration blocks write") and, worse, lets a
    # failed task silently retry a non-idempotent INSERT — confirmed live via
    # duplicate rows. Must be 1 for both the guard and correctness.
    .config('spark.task.maxFailures', '1')
    .getOrCreate()
)
spark.sql('SHOW NAMESPACES').show()

In [ ]:
spark.sql(f"SHOW TABLES IN {DB}").show(truncate=False)

# SHOW TABLE EXTENDED is not supported for v2 tables, since the
# Polaris/Iceberg REST catalog is a v2 catalog and that legacy command
# only supports v1/Hive-style catalogs.
NAMESPACE = 'demo'

rows = []
for t in spark.sql(f"SHOW TABLES IN {NAMESPACE}").collect():
    try:
        desc = spark.sql(f"DESCRIBE TABLE EXTENDED {NAMESPACE}.{t.tableName}").collect()
        location = next(r.data_type for r in desc if r.col_name == 'Location')
    except Exception:
        # Alice/Bob don't have grants on every table in this namespace —
        # DESCRIBE also needs LOAD_TABLE_WITH_READ_DELEGATION (see runbook
        # bug #4), so it 403s here exactly like a real read/write would.
        location = '<no access>'
    rows.append((NAMESPACE, t.tableName, location))

spark.createDataFrame(rows, ['namespace', 'table', 'location']).show(truncate=False)

In [ ]:
spark.sql('''
  CREATE TABLE IF NOT EXISTS demo.products (
    product_id INT, name STRING, price DOUBLE
  ) USING iceberg
''')
spark.sql('''
  CREATE TABLE IF NOT EXISTS demo.events (
    event_id INT, description STRING
  ) USING iceberg
''')
spark.sql('''
  CREATE TABLE IF NOT EXISTS demo.order_items (
    order_id INT, item STRING, qty INT, written_by STRING
  ) USING iceberg
''')

spark.sql("INSERT INTO demo.products VALUES (1, 'Widget', 9.99), (2, 'Gadget', 19.99)")
spark.sql("INSERT INTO demo.events VALUES (1, 'restock')")
spark.sql("INSERT INTO demo.order_items VALUES (101, 'gadget', 1, 'bob')")

print('Bob-owned tables:')
spark.sql('SELECT * FROM demo.products').show()
spark.sql('SELECT * FROM demo.events').show()

In [ ]:
# demo.order_items is the deliberate overlap with Alice — this should show
# Alice's earlier row (written_by='alice') alongside Bob's own.
spark.sql('SELECT * FROM demo.order_items ORDER BY order_id').show()

In [ ]:
# Pass/fail auth check: Bob has no grant on demo.customers, so this write
# must be REJECTED by Polaris.
#
# The except clause is deliberately narrow. `except Exception` printed PASS
# for *any* failure, including a NameError from a session that never built
# — which is exactly how an expired Polaris token presents (runbook.md
# step 8). A gate that passes while the demo is broken is worse than no
# gate, so a PASS here requires Polaris's own refusal: the right exception
# type *and* a message naming the denied op. Anything else propagates and
# fails the cell loudly.
from py4j.protocol import Py4JJavaError
from pyspark.errors import AnalysisException

DENIAL = ('ForbiddenException', 'not authorized for op')

try:
    spark.sql('CREATE TABLE IF NOT EXISTS demo.customers (customer_id INT, name STRING) USING iceberg')
    spark.sql("INSERT INTO demo.customers VALUES (999, 'should not be allowed')")
    print('FAIL: Bob was able to write to demo.customers — isolation is broken')
except (Py4JJavaError, AnalysisException) as e:
    if not all(marker in str(e) for marker in DENIAL):
        raise  # a real breakage, not the authorization refusal under test
    print('PASS: write to demo.customers correctly rejected by Polaris')
    print(type(e).__name__, str(e)[:300])

## VASTDB extension

The `whitmore` bucket is also a VAST Database. These tables (`shipments`,
`reviews`, `inventory_events`) are registered in Polaris as **Generic
Tables**, which is discovery only — Polaris 1.7.0 does not vend credentials
for them (checked upstream: the vending wire format exists in the OpenAPI
spec but the server handler never populates it, and a load returns
`storage-access-configs: []`).

So the two halves of this notebook split cleanly:

- **Polaris answers "what tables exist and where"** — the cell above reads
  bucket, schema and endpoints out of each registration, gated per table by
  this principal's `TABLE_FULL_METADATA` grant.
- **VAST answers "may you read or write it"** — the cells below go through
  the VAST DB Spark connector (`ndb`), authenticated with bob-db's own
  separate, narrowly-scoped key. Isolation here is enforced by VAST's
  Identity Policy on that key, not by Polaris.

The last cell shows both layers refusing the same table independently.

In [ ]:
spark.sql(f"SHOW TABLES IN {DB}").show()

spark.sql(f"INSERT INTO {ndb('reviews')} VALUES (1, 1, 5, 'Great widget'), (2, 2, 4, 'Solid gadget')")
spark.sql(f"INSERT INTO {ndb('inventory_events')} VALUES (2, 'gadget', 1, 'bob')")

print('bob-owned VASTDB tables:')
spark.sql(f"SELECT * FROM {ndb('reviews')}").show()

In [ ]:
# inventory_events is the deliberate VASTDB-side overlap with alice — run
# this cell again after Alice.ipynb has run to see their row appear too.
spark.sql(f"SELECT * FROM {ndb('inventory_events')} ORDER BY event_id").show()

In [ ]:
# Pass/fail auth check, now at both layers independently.
#
# Layer 1 — Polaris already refused to hand over the metadata for
# shipments in the discovery cell, because bob's catalog role has no
# TABLE_FULL_METADATA grant on it.
assert 'shipments' in denied, "expected Polaris to deny metadata for shipments"
print('PASS (Polaris): metadata load for shipments was refused — it is not in `registered`')

# Layer 2 — and VAST refuses the write regardless, because bob-db's
# Identity Policy grants no access to it. This is the layer that actually
# protects the data: Polaris does not gate VASTDB reads/writes at all, so
# the identifier has to be built by hand here — `ndb()` cannot produce it.
#
# Narrow, for the reason given in the Iceberg check above. VAST's refusal
# reaches Spark as TABLE_OR_VIEW_NOT_FOUND rather than a permission error:
# the connector 403s on LIST_COLUMNS while resolving the name (the
# VastClient line in the driver log) and then reports the name as
# unresolvable. So accept that, or an explicit denial if a future
# connector surfaces one — but require the table name either way, and let
# anything else fail the cell.
from py4j.protocol import Py4JJavaError
from pyspark.errors import AnalysisException

FORBIDDEN_TABLE = 'shipments'
DENIAL = ('TABLE_OR_VIEW_NOT_FOUND', 'AccessDenied', 'Forbidden', 'not allowed', '403')

forbidden = f"{DB}.{FORBIDDEN_TABLE}"
try:
    spark.sql(f"INSERT INTO {forbidden} VALUES (999, 999, 'should not be allowed')")
    print('FAIL: bob was able to write to shipments — isolation is broken')
except (Py4JJavaError, AnalysisException) as e:
    msg = str(e)
    if FORBIDDEN_TABLE not in msg or not any(marker in msg for marker in DENIAL):
        raise  # a real breakage, not VAST's refusal under test
    print('PASS (VAST): write to shipments correctly rejected')
    print(type(e).__name__, msg[:300])